# 🤖 تحليل `ForexAgent` — Sprint 9، الخطوة 1

---

## 🎯 TL;DR

> **`ForexAgent`** هو **المنسق الأعلى**: يجمع `ForexService` (سعر + مؤشرات) + `ForecastService` (ML + XAI) → **`AgentAssessment`**: قرار (BUY/SELL/WAIT) + ثقة + Market Regime + أسباب + سرد. **الفكرة**: Confluence (توافق الإشارات) = قرار أقوى.

---

## 📊 المخطط المعماري

```mermaid
graph TD
    A[ForexAgent] --> B[ForexService]
    A --> C[ForecastService]
    B --> D[get_current_price]
    B --> E[analyze_pair - SMA, RSI, MACD]
    C --> F[generate_forecast + explain]
    D --> G[_synthesize_analysis]
    E --> G
    F --> G
    G --> H[AgentAssessment]
    H --> I[BUY / SELL / WAIT]
    H --> J[confidence 0-85%]
    H --> K[Market Regime]
    H --> L[key_reasons]
    H --> M[narrative]

    style A fill:#bfb,stroke:#333,stroke-width:3px
    style G fill:#bdf,stroke:#333,stroke-width:2px
    style H fill:#f9f,stroke:#333,stroke-width:2px
```

---

## 🔬 التشريح

### 1️⃣ `AgentAssessment` — Value Object (10 حقول)

| الفئة | الحقول |
|-------|--------|
| **السياق** | `pair`, `interval`, `current_price` |
| **القرار** | `recommended_action`, `confidence_score` |
| **التصنيف** | `market_regime`, `technical_bias`, `ml_bias` |
| **التفسير** | `key_reasons: list[str]`, `narrative: str` |

**🎯 `to_dict`:** تحويل لـ JSON/CLI — 10 حقول كلها serializable.

---

### 2️⃣ `analyze_market` — 3 مراحل

```mermaid
sequenceDiagram
    participant A as Agent
    participant F as ForexService
    participant M as ForecastService

    A->>F: get_current_price(pair)
    F-->>A: Decimal
    A->>F: analyze_pair(indicators=[sma,rsi,macd])
    F-->>A: tech_data
    A->>M: generate_forecast(explain=True)
    M-->>A: ForecastReport
    A->>A: _synthesize_analysis(...)
    A-->>A: AgentAssessment
```

**🎯 تريكة `explain=True`:** نطلب XAI مباشرة — نحتاجه للأسباب.

---

### 3️⃣ `_synthesize_analysis` — **قلب الوكيل**

#### المرحلة أ: المؤشرات الفنية (3 إشارات)

```mermaid
graph TD
    A[tech_data.indicators] --> B[SMA]
    A --> C[RSI]
    A --> D[MACD]
    B --> E{price > sma?}
    E -->|Yes| F[bullish +1]
    E -->|No| G[bearish +1]
    C --> H{RSI level}
    H -->|>70| I[bearish +1 - overbought]
    H -->|<30| J[bullish +1 - oversold]
    H -->|>=50| K[bullish +1 - momentum]
    H -->|<50| L[bearish +1 - weak]
    D --> M{hist > 0?}
    M -->|Yes| N[bullish +1]
    M -->|No| O[bearish +1]

    style F fill:#bbf,stroke:#333
    style J fill:#bbf,stroke:#333
    style K fill:#bbf,stroke:#333
    style N fill:#bbf,stroke:#333
    style G fill:#fdd,stroke:#333
    style I fill:#fdd,stroke:#333
    style L fill:#fdd,stroke:#333
    style O fill:#fdd,stroke:#333
```

**جدول إشارات المؤشرات:**

| المؤشر | الشرط | النقطة | السبب |
|--------|-------|--------|-------|
| **SMA** | price > sma | +1 bullish | اتجاه صاعد |
| SMA | price < sma | +1 bearish | اتجاه هابط |
| **RSI** | > 70 | +1 bearish | Overbought (انعكاس محتمل) |
| RSI | < 30 | +1 bullish | Oversold (ارتداد محتمل) |
| RSI | 50-70 | +1 bullish | زخم إيجابي |
| RSI | 30-50 | +1 bearish | زخم ضعيف |
| **MACD** | hist > 0 | +1 bullish | زخم صاعد |
| MACD | hist < 0 | +1 bearish | زخم هابط |

**🎯 تريكة RSI 4 حالات:** التطبيع المزدوج — Overbought = **إشارة هبوط** (عكس المتوقع!)، Oversold = **إشارة صعود** (mean reversion).

**النتيجة:** `technical_bias` = BULLISH / BEARISH / NEUTRAL (بناءً على عدد النقاط).

---

#### المرحلة ب: ML + XAI

```python
if forecast.direction == "UP" and forecast.up_probability >= 55.0:
    bullish_points += 2       # ← وزن مضاعف
    reasons.append(f"ML predicts UP with {prob:.1f}%")
elif forecast.direction == "DOWN" and forecast.down_probability >= 55.0:
    bearish_points += 2
else:
    reasons.append("ML probability is neutral...")
```

**🎯 تريكة `+2`:** ML **أثقل وزناً** من مؤشر فني واحد (لأنه يجمع 5 features).

**🎯 عتبة 55%:** لا نأخذ إشارة ضعيفة (< 55%) — **Fail Soft**.

**XAI:**
```python
if forecast.explanation:
    if UP and top_bullish_factors:
        reasons.append(f"Top XAI Driver: {driver.name} ...")
    elif DOWN and top_bearish_factors:
        reasons.append(f"Top XAI Drag: {drag.name} ...")
```

**🎯 تريكة:** نأخذ **أقوى عامل فقط** — لا نُثقل الأسباب.

---

#### المرحلة ج: القرار + الثقة

```mermaid
graph TD
    A[net_bias = bull - bear] --> B{net >= 2 AND bull >= 3?}
    B -->|Yes| C[BUY]
    C --> C1[regime = Bullish Expansion]
    C1 --> C2[confidence = min 85, 50 + bull/total × 40]
    B -->|No| D{net <= -2 AND bear >= 3?}
    D -->|Yes| E[SELL]
    E --> E1[regime = Bearish Correction]
    E1 --> E2[confidence = min 85, 50 + bear/total × 40]
    D -->|No| F[WAIT]
    F --> F1[regime = Consolidation]
    F1 --> F2[confidence = 50.0]

    style C fill:#bfb,stroke:#333,stroke-width:2px
    style E fill:#fdd,stroke:#333,stroke-width:2px
    style F fill:#ffd,stroke:#333,stroke-width:2px
```

**شرطان للقرار:**

| الشرط | المعنى |
|-------|--------|
| `net_bias >= 2` | فرق 2 نقاط على الأقل لصالح الصعود |
| `bullish_points >= 3` | 3 إشارات صاعدة على الأقل (من أصل 5) |

**🎯 تريكة الشرطين:** لا نكتفي بـ `net` فقط — نضمن **قوة مطلقة** أيضاً. مثال: `bull=3, bear=1` → `net=2` → **BUY**. لكن `bull=1, bear=0` → `net=1` → **WAIT**.

**معادلة الثقة:**
```
confidence = min(85, 50 + (bullish/total) × 40)
```

**أمثلة:**

| bull | bear | total | نسبة | confidence |
|------|------|-------|------|------------|
| 3 | 1 | 4 | 75% | 50 + 30 = **80%** |
| 4 | 1 | 5 | 80% | 50 + 32 = **82%** |
| 5 | 0 | 5 | 100% | 50 + 40 = **90% → 85%** (سقف) |

**🎯 سقف 85%:** لا نوعد بـ > 85% — **تواضع إحصائي**.

**WAIT = 50%:** قرار محايد — **Confidence ليس له معنى**.

---

#### المرحلة د: السرد (Narrative)

```python
narrative = (
    f"Market for {pair} is currently in a {market_regime}. "
    f"Technical indicators lean {tech_bias.lower()}, while the predictive ML model signals {ml_bias}. "
    f"Given the confluence of signals, the recommended tactical posture is to {action} with {confidence:.0f}% confidence."
)
```

**🎯 3 جمل:** الحالة + الانحياز + القرار. **بشرية وقابلة للقراءة**.

---

## 📊 تدفق الإشارات (Signal Confluence)

```mermaid
graph LR
    A[SMA] -->|+1| T[total]
    B[RSI] -->|+1| T
    C[MACD] -->|+1| T
    D[ML] -->|+2| T
    T --> E{bull >= 3 AND net >= 2?}
    E -->|Yes| F[BUY]
    E -->|No| G{bear >= 3 AND net <= -2?}
    G -->|Yes| H[SELL]
    G -->|No| I[WAIT]

    style F fill:#bfb,stroke:#333,stroke-width:3px
    style H fill:#fdd,stroke:#333,stroke-width:3px
    style I fill:#ffd,stroke:#333,stroke-width:3px
```

**الحد الأقصى للنقاط:** 3 (Tech) + 2 (ML) = **5 نقاط**.

**الحد الأدنى للقرار:** 3 نقاط من 5 = **60%**.

---

## 🧪 التحقق (وصف)

**السيناريو 1 — صاعد:**
- SMA: price > sma → +1 bull
- RSI: 60 → +1 bull
- MACD: hist > 0 → +1 bull
- ML: UP 62% → +2 bull
- **النتيجة:** 5 bull, 0 bear → **BUY 85%** (سقف)

**السيناريو 2 — مختلط:**
- SMA: price < sma → +1 bear
- RSI: 55 → +1 bull
- MACD: hist < 0 → +1 bear
- ML: neutral → 0
- **النتيجة:** 1 bull, 2 bear → **WAIT 50%**

---

## ⚠️ ملاحظات على التصميم

| النقطة | التأثير |
|--------|---------|
| ML +2 vs Tech +1 | ML يسيطر بسهولة — **توازن غير مثالي** |
| RSI 50-70 → bullish | RSI 65 قد يكون متضخماً — **قد يضلل** |
| `limit=120` default | للـ forecast فقط — tech_data يستخدم نفس limit |
| سقف 85% | جيد — لكن لا يوجد "very strong" |
| لا `stop_loss` / `take_profit` | لا نقاط دخول/خروج |
| `total_points == 0` (نادراً) | `division by zero` — **يحتاج حماية** |

---

## 🎯 القيمة المعمارية

| المبدأ | التطبيق |
|--------|---------|
| **Orchestration** | يجمع 3 خدمات |
| **Signal Confluence** | توافق الإشارات = قرار |
| **Weighted Voting** | ML أثقل من Tech |
| **Fail Soft** | ML < 55% = لا صوت |
| **Safeguard** | `net >= 2` + `points >= 3` |
| **Confidence Cap** | 85% (لا مبالغة) |
| **Narrative** | 3 جمل بشرية |
| **XAI Integration** | `top_driver` في الأسباب |

---

## 🗺️ Sprint 9

```mermaid
graph LR
    A["agent/market_agent.py 🆕"] --> B["الخطوة 2: تحسين الثقة"]
    B --> C["الخطوة 3: CLI: forex agent analyze"]
    C --> D["الخطوة 4: tests + docs"]

    style A fill:#bfb,stroke:#333,stroke-width:3px
    style C fill:#ddd,stroke:#333
```

---

## 📋 الخطوات التالية

### الخطوة 2 — تحسين `confidence`
**المقترح:** تعديل معادلة الثقة:
```python
confidence = min(90, 50 + (net_bias / total_points) * 50)
```

**الفوائد:**
- نسبة **صافية** (net) بدل **إجمالية** (bull).
- سقف أعلى (90%).
- `total_points = 0` → `confidence = 50` (حماية).

### الخطوة 3 — CLI
```bash
forex agent analyze --pair EUR/USD --interval 1day --limit 120
```

**Output متوقع:**

```
========================================================
 🤖 Forex AI Agent — Market Analysis
========================================================
Pair          : EUR/USD [1day]
Current Price : 1.0842
Market Regime : Bullish Expansion

📊 Technical Bias : BULLISH
🤖 ML Bias        : UP

💡 Recommended Action : 🟢 BUY (Confidence: 82%)

--- Key Reasons ---
  1. Price (1.08420) is trading above 20-period SMA (1.07950).
  2. RSI shows positive momentum at 58.4.
  3. MACD Histogram is positive (+0.00044).
  4. ML Model predicts UP direction with 62.5% probability.
  5. Top XAI Driver: RSI (Normalized) is supporting upward movement.

--- Narrative ---
Market for EUR/USD is currently in a Bullish Expansion.
Technical indicators lean bullish, while the predictive ML
model signals UP. Given the confluence of signals, the
recommended tactical posture is to BUY with 82% confidence.
========================================================
```

**السؤال الجوهري:** كيف نُظهر **لماذا WAIT**؟

**الجواب:** `key_reasons` **تُسرد دائماً** — حتى لو `WAIT`. المتداول يرى **السبب** ولا يبقى في حيرة.

In [1]:
from agent.market_agent import ForexAgent

agent = ForexAgent()
try:
    report = agent.analyze_market(pair='EUR/USD', interval='1day', limit=100)
    print('Action     :', report.recommended_action)
    print('Confidence :', f'{report.confidence_score:.1f}%')
    print('Regime     :', report.market_regime)
    print('Narrative  :', report.narrative)
except Exception as e:
    print('Note:', e)

[2026-11-07 00:11:58] [INFO] [forex_tracker]: Connected to SQLite database at: ./data\candles.db
[2026-11-07 00:11:58] [WARNING] [forex_tracker]: Redis is not available (unknown command 'HELLO'). Falling back to MemoryCache.
[2026-11-07 00:11:58] [INFO] [forex_tracker]: Connected to SQLite database at: ./data\candles.db
[2026-11-07 00:11:58] [INFO] [forex_tracker]: 🌐 [Cache Miss] جلب السعر من TwelveData لزوج EUR/USD...
[2026-11-07 00:11:58] [INFO] [forex_tracker]: جلب 1 شمعة لزوج EUR/USD بإطار 1day من TwelveData...
[2026-11-07 00:11:58] [INFO] [forex_tracker]: إرسال طلب HTTP GET إلى: https://api.twelvedata.com/time_series
[2026-11-07 00:11:59] [INFO] [forex_tracker]: تم بنجاح استخراج 1 شمعة لزوج EUR/USD.
[2026-11-07 00:11:59] [INFO] [forex_tracker]: Forecast generated for EUR/USD [1day]: Direction=DOWN, P(Up)=37.8%, TestAcc=42.86%


Note: 'IndicatorResult' object has no attribute 'histogram'
